# Does a coupled continuous CNN improve attention under temporal extrapolation?

*Executable synthetic study — academic research format — PyTorch, NumPy, SciPy — Google Colab*

## Abstract

Attention-based representations of irregularly sampled trajectories can provide strong short-horizon predictions, yet may generalize poorly to forecast intervals outside their training distribution. We investigate whether a structured, continuous-time convolutional evolution module offers complementary inductive bias when attached to an attention encoder. We construct two analytically solvable dynamical systems: a spatially local, diffusively coupled system with a global latent variable, and a nonlocal stable system in which the diffusion assumption is violated. Given noisy observations at irregular times, the model predicts a future spatial state at a supplied, variable forecast horizon. We compare horizon-aware attention alone, a one-step convolutional Euler update, four-step coupled continuous convolution, a dense neural ODE, and structural ablations. The study evaluates interpolation and extrapolation error, nonlocal failure behavior, parameter counts, and training/inference time. We hypothesize that structured continuous dynamics will improve temporal extrapolation when the data support their locality assumptions, but may fail when those assumptions are incorrect. Results are computed when this notebook runs; no advantage is presumed in advance.

## 1. Research questions

**RQ1.** Does attention augmented with continuous coupled local dynamics extrapolate more accurately to longer unseen time intervals than horizon-aware attention alone?

**RQ2.** Does any gain arise from spatial coupling, the global ODE, or continuous integration itself? A dense Neural ODE provides a critical control.

**RQ3.** Is the advantage robust to violation of the locality/diffusion assumption?

**Pre-registered decision:** CCNN earns a narrow *positive* result only if it improves held-out long-gap error compared with attention and the single-step Euler baseline *on the local system*. It earns an architectural-specific advantage only if it also improves over the dense Neural ODE after considering runtime. If CCNN loses on the nonlocal control, report this failure rather than hiding it.


## 2. Proposed method and expected outcomes

Let \(u(t)\in\mathbb R^{10}\) be a ring of spatially coupled channels and \(z(t)\in\mathbb R\) a global latent state. A multi-head attention encoder processes seven noisy observations \((\tilde u(t_i),t_i)\) and estimates an initial state \((\hat u(t_7),\hat z(t_7))\). A structured continuous-time module evolves both states through the **actual future time gap** \(\Delta t\):

\[
\dot u = D_\theta\Delta_{\mathrm{ring}}u-\lambda_\theta u+b_\theta z + C_\theta(u),
\qquad
\dot z=-\gamma_\theta z+\kappa_\theta\operatorname{mean}(u).
\]

The CCNN update uses four RK4 steps and evolves \((u,z)\) together. The attention-only control *also receives* \(\Delta t\) and \(\Delta t^2\), avoiding an unfair missing-time-input advantage. The generic Neural ODE uses dense learned dynamics without the explicit spatial operator. The Euler control uses the CCNN's same vector field but applies one forward-Euler update.

**Expected outcome (hypothesis, not measured result):** continuous integration may help when test gaps are substantially longer than training gaps; a shared local operator may help when the true process is local. Dense dynamics may be equally effective; nonlocal dynamics may penalize CCNN's inductive bias. Computation may be slower than attention alone.

### Proposed method diagram

The following generated graph explicitly emphasizes the CCNN contribution. It distinguishes the common attention encoder from alternative rollout mechanisms.


In [ ]:
# Generated diagram (Graphviz is available in hosted Colab runtimes).
from IPython.display import display
try:
    from graphviz import Digraph
    fig = Digraph('proposed_method', graph_attr={'rankdir':'LR'})
    fig.node('data','7 irregular noisy observations + times',shape='box')
    fig.node('enc','Shared attention encoder',shape='box')
    fig.node('ccnn','PROPOSED: coupled CCNN\n4-step RK4 on (u,z)',shape='box',style='bold,rounded',penwidth='3')
    fig.node('base','Controls: direct attention / Euler / dense Neural ODE',shape='box')
    fig.node('test','Same held-out future state and time gap',shape='box')
    fig.edge('data','enc');fig.edge('enc','ccnn');fig.edge('enc','base')
    fig.edge('ccnn','test');fig.edge('base','test')
    display(fig)
except Exception as e:
    print('Optional Graphviz preview unavailable:',type(e).__name__,str(e)[:120])


## 3. Implementation — environment and reproducible configuration

The experiment uses CPU by default for reproducibility and because its models are small. A standard Colab runtime is sufficient. Select `QUICK_TEST=True` to verify the notebook before running the **full three-seed experiment**. The original benchmark used three seeds, 24 epochs, 900 training trajectories per seed, 220 validation trajectories, and 320 test trajectories per condition. Setting `QUICK_TEST=False` restores these values. Dependencies: PyTorch, NumPy, SciPy, pandas, and Matplotlib; no Hugging Face token is required for this toy theory test.


In [ ]:
# Run this cell only if your Colab runtime is missing one of these packages.
# %pip -q install torch numpy scipy pandas matplotlib graphviz
import torch, numpy as np, scipy.linalg as la, time, json, os, copy
from torch import nn
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

QUICK_TEST = False
SEEDS = [1] if QUICK_TEST else [1, 2, 3]
TRAIN_EPOCHS = 3 if QUICK_TEST else 24
TRAIN_EXAMPLES = 180 if QUICK_TEST else 900
VAL_EXAMPLES = 80 if QUICK_TEST else 220
TEST_EXAMPLES = 120 if QUICK_TEST else 320
OUTDIR = Path('/content/ccnn_attention_toy_outputs') if Path('/content').exists() else Path.cwd()/'ccnn_attention_toy_outputs'
OUTDIR.mkdir(parents=True,exist_ok=True)
torch.set_num_threads(2)
print('Configuration:',dict(seeds=SEEDS,epochs=TRAIN_EPOCHS,training_trajectories=TRAIN_EXAMPLES,outdir=str(OUTDIR)))


## 4. Synthetic dynamical systems and data generation

We generate reference trajectories analytically using the matrix exponential of a stable linear system. The local system has ring diffusion and coupling to an unobserved global scalar. The **negative control** uses a nonlocal skew-symmetric mixing matrix with stabilizing decay and no local spatial diffusion. Both systems yield noisy, irregularly timed observations. Ground-truth parameters are not supplied to the neural models.

Training forecast intervals are sampled uniformly from **0.08–0.30** time units; extrapolation intervals are sampled from **0.35–0.95**. Distinct random states produce training, validation, and test examples. The encoder receives observation timestamps and inter-observation intervals, and every method receives the future forecast interval.


In [ ]:
import torch, numpy as np, scipy.linalg as la, time, json, sys, os
from torch import nn
import torch.nn.functional as F

torch.set_num_threads(2)
N=10; HIST=7; WIDTH=32
# True linear coupled PDE-ODE dynamics for a ring of 10 spatial channels.
L=np.zeros((N,N))
for i in range(N):
    L[i,(i-1)%N]=1;L[i,i]=-2;L[i,(i+1)%N]=1
G=np.zeros((N+1,N+1))
G[:N,:N]=0.85*L-0.12*np.eye(N)
G[:N,N]=0.45*(1+0.35*np.cos(np.arange(N)*2*np.pi/N))
G[N,:N]=.50/N
G[N,N]=-.75
# Negative control: a nonlocal stable dynamical system without spatial diffusion.
DYNAMICS='local_coupled'
if DYNAMICS=='nonlocal':
    rng_nonlocal=np.random.default_rng(891)
    R=rng_nonlocal.normal(size=(N,N))
    G[:N,:N]=-.35*np.eye(N)+.85*(R-R.T)/np.sqrt(2*N)
    G[:N,N]=.4*rng_nonlocal.normal(size=N)
    G[N,:N]=.3*rng_nonlocal.normal(size=N)/N
    G[N,N]=-.6
w,V=np.linalg.eig(G)
Vi=np.linalg.inv(V)

def sample(num,seed,shift=False,uniform=False,obs_stretch=False):
    rng=np.random.default_rng(seed)
    # Per recording distinct state; no overlap between splits
    base=rng.normal(size=(num,3))
    x=np.arange(N)[None,:]
    u0=(base[:,0,None]*np.sin(2*np.pi*x/N)+base[:,1,None]*np.cos(4*np.pi*x/N)
        +0.6*rng.normal(size=(num,N)))
    z0=base[:,2,None]
    y0=np.concatenate([u0,z0],axis=1)
    coeff=y0@Vi.T
    if uniform:
        times=np.repeat(np.arange(1,HIST+1,dtype=float)[None,:]*0.18,num,axis=0)
    else:
        gaps=rng.uniform(0.05,0.75 if obs_stretch else 0.38,size=(num,HIST))
        times=np.cumsum(gaps,axis=1)
    dt=rng.uniform(.35,.95,size=num) if shift else rng.uniform(.08,.30,size=num)
    all_t=np.concatenate([times,(times[:,-1]+dt)[:,None]],axis=1)
    expw=np.exp(all_t[:,:,None]*w[None,None,:])
    full=np.einsum('btk,bk,lk->btl',expw,coeff,V).real
    obs=full[:,:HIST,:N]+rng.normal(0,.07,size=(num,HIST,N))
    target=full[:,-1,:N]
    return tuple(torch.tensor(a,dtype=torch.float32) for a in (obs,times,dt[:,None],target))


# Switch between the two reference dynamical systems without changing the model code.
def set_reference_dynamics(mode):
    global DYNAMICS, G, w, V, Vi
    if mode not in ('local_coupled','nonlocal'):
        raise ValueError('mode must be local_coupled or nonlocal')
    DYNAMICS = mode
    G = np.zeros((N+1,N+1))
    if mode == 'local_coupled':
        G[:N,:N] = .85*L - .12*np.eye(N)
        G[:N,N] = .45*(1+.35*np.cos(np.arange(N)*2*np.pi/N))
        G[N,:N] = .50/N
        G[N,N] = -.75
    else:
        rng_nonlocal = np.random.default_rng(891)
        R = rng_nonlocal.normal(size=(N,N))
        G[:N,:N] = -.35*np.eye(N) + .85*(R-R.T)/np.sqrt(2*N)
        G[:N,N] = .4*rng_nonlocal.normal(size=N)
        G[N,:N] = .3*rng_nonlocal.normal(size=N)/N
        G[N,N] = -.6
    w,V=np.linalg.eig(G)
    Vi=np.linalg.inv(V)

for domain in ('local_coupled','nonlocal'):
    set_reference_dynamics(domain)
    train=sample(4,12)
    assert all(t.isfinite().all().item() for t in train),domain
    print(domain, 'sample batch shapes:',[tuple(x.shape) for x in train])
set_reference_dynamics('local_coupled')


## 5. Candidate models: common attention encoder, different evolution rules

All methods use the same attention encoder and noisy observations. Models differ only in how the estimated state is advanced to the requested future time: direct horizon-aware regression, a one-step local Euler update, a four-step structured CCNN/RK4 solve, or a four-step dense neural ODE solve. The CCNN ablations remove diffusion or global-to-local coupling. Parameter counts are reported because these models are **not exactly parameter-matched**.


In [ ]:
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb=nn.Linear(N+2,WIDTH)
        self.attn=nn.MultiheadAttention(WIDTH,4,batch_first=True)
        self.norm=nn.LayerNorm(WIDTH)
        self.den=nn.Linear(WIDTH,N)
        self.zinit=nn.Linear(WIDTH,1)
    def forward(self,x,t):
        diff=torch.cat([t[:,:1],t[:,1:]-t[:,:-1]],dim=1)
        a=self.emb(torch.cat([x,t[:,:,None],diff[:,:,None]],dim=-1))
        att,_=self.attn(a,a,a,need_weights=False)
        h=self.norm(a+att)[:,-1]
        u=x[:,-1]+.12*torch.tanh(self.den(h))
        z=self.zinit(h)
        return u,z,h

class Toy(nn.Module):
    def __init__(self,kind):
        super().__init__()
        self.kind=kind
        self.enc=Encoder()
        if kind=='attention':
            # A stronger, horizon-aware attention baseline with more parameters.
            self.net=nn.Sequential(nn.Linear(N+WIDTH+2,64),nn.Tanh(),nn.Linear(64,N))
        else:
            if kind=='neural_ode':
                self.dense=nn.Sequential(nn.Linear(N+1,32),nn.Tanh(),nn.Linear(32,N+1))
            self.conv=nn.Conv1d(1,1,3,padding=0,bias=False)
            self.rawd=nn.Parameter(torch.tensor(-0.2))
            self.rawdecay=nn.Parameter(torch.tensor(-1.4))
            self.coupling=nn.Parameter(torch.ones(N)*.15)
            self.zdecay=nn.Parameter(torch.tensor(-.7))
            self.zdrive=nn.Parameter(torch.tensor(.25))
    def dyn(self,u,z):
        if self.kind=='neural_ode':
            dz=self.dense(torch.cat([u,z],dim=-1))
            return dz[:,:N],dz[:,N:]
        lap=torch.roll(u,-1,-1)-2*u+torch.roll(u,1,-1)
        # Periodic 3-point convolution, with no zero-padding boundary artifact.
        a=self.conv.weight[0,0,0]*torch.roll(u,1,-1)+self.conv.weight[0,0,1]*u+self.conv.weight[0,0,2]*torch.roll(u,-1,-1)
        du=(F.softplus(self.rawd) if self.kind!='ccnn_nodiff' else 0)*lap-F.softplus(self.rawdecay)*u + self.coupling*(z if self.kind!='ccnn_noglobal' else 0) +.15*torch.tanh(a)
        dz=-F.softplus(self.zdecay)*z+self.zdrive*u.mean(-1,keepdim=True)
        return du,dz
    def forward(self,x,t,dt):
        u,z,h=self.enc(x,t)
        if self.kind=='attention':
            return u+dt*self.net(torch.cat([u,h,dt,dt.square()],dim=-1))
        if self.kind=='euler':
            du,dz=self.dyn(u,z)
            return u+dt*du
        # 4-step full-horizon RK4 integration, z persists.
        hstep=dt/4
        for k in range(4):
            a,b=self.dyn(u,z)
            c,d=self.dyn(u+.5*hstep*a,z+.5*hstep*b)
            e,f=self.dyn(u+.5*hstep*c,z+.5*hstep*d)
            g,j=self.dyn(u+hstep*e,z+hstep*f)
            u=u+hstep*(a+2*c+2*e+g)/6
            z=z+hstep*(b+2*d+2*f+j)/6
        return u



## 6. Optimization and independently held-out evaluation

Each model is trained with AdamW and early checkpoint selection using validation MSE; training never sees long-gap test labels. Four held-out domains measure normal prediction, long-gap extrapolation, regular observation timing, and stretched irregular observation timing. Report mean **and standard deviation** across seeds. The experiment is small and does not support broad statistical significance claims.


In [ ]:
def train(kind,seed):
    torch.manual_seed(seed)
    m=Toy(kind)
    opt=torch.optim.AdamW(m.parameters(),lr=.004)
    train=sample(TRAIN_EXAMPLES,seed+11)
    val=sample(VAL_EXAMPLES,seed+101)
    train_batches=10
    best=1e9;beststate=None
    start=time.perf_counter()
    for epoch in range(TRAIN_EPOCHS):
        m.train()
        idx=torch.randperm(len(train[0]))
        for inds in idx.split(120):
            opt.zero_grad()
            pred=m(*(a[inds] for a in train[:3]))
            loss=F.mse_loss(pred,train[3][inds]);loss.backward()
            nn.utils.clip_grad_norm_(m.parameters(),2)
            opt.step()
        m.eval()
        with torch.no_grad():
            pred=m(*val[:3]);v=F.mse_loss(pred,val[3]).item()
        if v<best:
            best=v;beststate={k:p.detach().clone() for k,p in m.state_dict().items()}
    m.load_state_dict(beststate)
    out={'kind':kind,'seed':seed,'val':best,'train_seconds':time.perf_counter()-start,'parameters':sum(p.numel() for p in m.parameters())}
    m.eval()
    with torch.no_grad():
        for domain,kwargs in [('iid',{}),('long_gap',{'shift':True}),('regular',{'uniform':True}),('stretched_observations',{'obs_stretch':True})]:
            ex=sample(TEST_EXAMPLES,seed+1001,**kwargs)
            start_inf=time.perf_counter()
            out[domain]=F.mse_loss(m(*ex[:3]),ex[3]).item()
            if domain=='long_gap':out['inference_ms_320']=1000*(time.perf_counter()-start_inf)
    return out



## 7. Run A — local coupled dynamics (the matched-inductive-bias test)

This test is intentionally favorable to CCNN: the synthetic system has a local ring Laplacian and a global latent variable. Report whether explicit locality and continuous integration improve predictions over the other arms. Improvements in this test alone cannot establish generality.


In [ ]:
MODEL_KINDS_LOCAL = ('attention','euler','ccnn','neural_ode','ccnn_nodiff','ccnn_noglobal')
local_results=[]
set_reference_dynamics('local_coupled')
for seed in SEEDS:
    for kind in MODEL_KINDS_LOCAL:
        result=train(kind,seed)
        local_results.append(result)
        print('Local',seed,kind,'IID',round(result['iid'],6),'long-gap',round(result['long_gap'],6))
(OUTDIR/'local_coupled.json').write_text(json.dumps(local_results,indent=2))
print('Saved local results:',OUTDIR/'local_coupled.json')


## 8. Run B — nonlocal negative control

The **same algorithms** are evaluated on a stable nonlocal system. CCNN's local diffusion is mismatched to these dynamics. A negative result is valuable: it tests whether any apparent gain is a genuine broadly useful mechanism or a consequence of matching the data generator. For efficiency, the two CCNN structural ablations are omitted here; attention, Euler, CCNN, and dense Neural ODE are compared.


In [ ]:
MODEL_KINDS_NONLOCAL = ('attention','euler','ccnn','neural_ode')
nonlocal_results=[]
set_reference_dynamics('nonlocal')
for seed in SEEDS:
    for kind in MODEL_KINDS_NONLOCAL:
        result=train(kind,seed)
        nonlocal_results.append(result)
        print('Nonlocal',seed,kind,'IID',round(result['iid'],6),'long-gap',round(result['long_gap'],6))
(OUTDIR/'nonlocal.json').write_text(json.dumps(nonlocal_results,indent=2))
print('Saved nonlocal results:',OUTDIR/'nonlocal.json')


## 9. Evaluation — comparative metrics and plots

A result table gives each method's mean and standard deviation across seeds. The plot separates in-distribution and long-gap errors. We emphasize **the proposed CCNN** with hatching, not by making its bar appear artificially best. A second plot compares train and inference time, since any quality improvement must be weighed against computation. All values below come from the code executed above; no benchmark values are hard-coded.


In [ ]:
results = pd.DataFrame(local_results + nonlocal_results)
results['dynamics']=(['local_coupled']*len(local_results)+['nonlocal']*len(nonlocal_results))
reported=['iid','long_gap','regular','stretched_observations','train_seconds','inference_ms_320','parameters']
summary=results.groupby(['dynamics','kind'])[reported].agg(['mean','std'])
print('MEASURED RESULTS (mean ± sample variability across seeds)')
display(summary.round(6))
results.to_csv(OUTDIR/'raw_per_seed.csv',index=False)
summary.to_csv(OUTDIR/'summary.csv')

fig,axes=plt.subplots(1,2,figsize=(15,5),sharey=False)
for ax,system in zip(axes,['local_coupled','nonlocal']):
    subset=results[results.dynamics==system]
    grouped=subset.groupby('kind').agg(iid=('iid','mean'),long_gap=('long_gap','mean'))
    labels=list(grouped.index)
    x=np.arange(len(labels));width=.38
    bars1=ax.bar(x-width/2,grouped.iid,width,label='IID')
    bars2=ax.bar(x+width/2,grouped.long_gap,width,label='Long gap')
    for j,name in enumerate(labels):
        if name=='ccnn':
            bars1[j].set_hatch('///');bars1[j].set_linewidth(2)
            bars2[j].set_hatch('///');bars2[j].set_linewidth(2)
    ax.set_xticks(x,labels,rotation=30,ha='right')
    ax.set_ylabel('Held-out MSE (lower better)')
    ax.set_title(system.replace('_',' ').title())
    ax.legend()
fig.suptitle('Proposed CCNN highlighted by hatching; not assumed best')
fig.tight_layout()
fig.savefig(OUTDIR/'quality_comparison.png',dpi=175)
plt.show()

fig,axes=plt.subplots(1,2,figsize=(14,4))
for ax,(metric,title) in zip(axes,[('train_seconds','Training time'),('inference_ms_320','Inference time for held-out batch')]):
    sub=results[results.dynamics=='local_coupled'].groupby('kind')[metric].mean().sort_values()
    bars=ax.bar(sub.index,sub.values)
    for b,kind in zip(bars,sub.index):
        if kind=='ccnn':b.set_hatch('///');b.set_linewidth(2)
    ax.set_title(title);ax.tick_params(axis='x',rotation=30)
fig.tight_layout()
fig.savefig(OUTDIR/'compute_comparison.png',dpi=175)
plt.show()


## 10. Evidence-based conclusion and future research

This section computes relative improvements from the **measured results**, explicitly comparing CCNN with attention and a generic Neural ODE. A conditional conclusion is printed rather than predetermining success. When an architectural advantage is absent, report the narrower finding that continuous integration or the matched spatial inductive bias helps under a particular distribution shift.


In [ ]:
def average_for(system,kind,metric):
    vals=results[(results.dynamics==system)&(results.kind==kind)][metric]
    return float(vals.mean())
for system in ['local_coupled','nonlocal']:
    a=average_for(system,'attention','long_gap')
    c=average_for(system,'ccnn','long_gap')
    d=average_for(system,'neural_ode','long_gap')
    print(f'{system}: CCNN long-gap change vs attention: {100*(a-c)/a:.2f}% lower error')
    print(f'{system}: CCNN long-gap change vs dense Neural ODE: {100*(d-c)/d:.2f}% lower error')
    print(f'{system}: CCNN wins vs BOTH attention and dense Neural ODE: {c<a and c<d}')
print('\nInterpretation: a three-seed toy result tests feasibility only. It is not a general attention improvement claim.')
print('Future research: spatial-resolution transfer (10→20/40 channels), exact memory retrieval,')
print('parameter matching, solver-budget sweeps, irregular missingness, and real recordings.')


## 11. Reproducibility artifacts and references

Rerunning saves raw per-seed JSON files, `raw_per_seed.csv`, `summary.csv`, and two figures under `ccnn_attention_toy_outputs/`. To retrieve them from Colab, download a ZIP assembled below. The toy currently uses PyTorch and explicit RK4 to isolate the hypothesis; moving to JAX/Diffrax is a separate engineering comparison, not evidence of a scientific gain.

**References**

1. M. Habiba and B. A. Pearlmutter, *Continuous Convolutional Neural Networks: Coupled Neural PDE and ODE* (2021), https://arxiv.org/abs/2111.00343.
2. A. Vaswani et al., *Attention Is All You Need* (2017), https://arxiv.org/abs/1706.03762.
3. R. T. Q. Chen et al., *Neural Ordinary Differential Equations* (2018), https://arxiv.org/abs/1806.07366.
4. A. Poli et al., *Hyena Hierarchy: Towards Larger Convolutional Language Models* (2023), https://arxiv.org/abs/2302.10866.
5. A. Gu and T. Dao, *Mamba: Linear-Time Sequence Modeling with Selective State Spaces* (2023), https://arxiv.org/abs/2312.00752.


In [ ]:
# Optional: zip your experimental outputs for a one-click Colab download.
import zipfile
archive=OUTDIR.parent/'ccnn_attention_toy_outputs.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as zf:
    for p in OUTDIR.glob('*'):
        if p.is_file():zf.write(p,arcname=p.name)
print('Created:',archive)
if 'google.colab' in __import__('sys').modules:
    from google.colab import files
    files.download(str(archive))
